
# CB với fav và rate
**Dùng Review (1..5) + Favorite (=5)**, đặc trưng **Tác giả/Thể loại/NXB**

Mục tiêu của notebook:
- Trình bày rõ từng bước để bạn có thể giải thích với giáo viên.
- **Không dùng TF‑IDF**; chỉ dùng **tập đặc trưng rời rạc** (authors, categories, publishers).
- **Ma trận W** đúng **thang 1..5** (favorite = 5).  
  - Nếu 1 user **đánh giá nhiều lần** cùng sách → **lấy review mới nhất** (`created_at DESC, id DESC`).
  - Nếu user **vừa review vừa favorite** → lấy **điểm lớn hơn** (tức là 5).
- Khi xuất **gợi ý** → **loại hết các sách user đã tương tác** (đã review/favorite).

Công thức chính:
- `X` (n_items × d): ma trận **Item–Feature** đã L2‑normalize từng sách.
- `W` (m_users × n_items): ma trận **Utility** (điểm 1..5 từ reviews/favorites).
- `P = W · X` (m × d): **hồ sơ user** (trung bình có trọng số các vector sách).
- `p̂_u = P[u] / ||P[u]||₂`: chuẩn hoá để dùng cosine ổn định.
- `score(u,i) = p̂_u · v_i` (vì `v_i` đã L2) ⇒ **cosine similarity**.
- **Top‑K** = sort score giảm dần và **bỏ các item đã tương tác** (ô `W[u,i] > 0`).



In [4]:

# ==============================
# CẤU HÌNH KẾT NỐI MYSQL
# ==============================
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "123456"

DB_BOOK  = "bookweb_book"
DB_ORDER = "bookweb_order"

# ==============================
# THAM SỐ MÔ HÌNH
# ==============================
# Tỷ trọng nhóm đặc trưng khi nối vector (dùng sqrt để scale)
LAMBDA_AUTHOR    = 0.4
LAMBDA_CATEGORY  = 0.4
LAMBDA_PUBLISHER = 0.2

# In minh hoạ
PRINT_MAX_ITEMS = 10
PRINT_MAX_USERS = 10
TOP_K = 10  # số gợi ý in ra mỗi user


In [5]:

import math, numpy as np, pandas as pd
from sqlalchemy import create_engine, text

def mysql_engine():
    """Tạo SQLAlchemy engine kết nối MySQL."""
    dsn = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}"
    return create_engine(dsn, pool_pre_ping=True, pool_recycle=1800)

def q_all(engine, sql, params=None):
    """Chạy SELECT và trả về list[dict]."""
    with engine.connect() as conn:
        res = conn.execute(text(sql), params or {})
        cols = res.keys()
        return [dict(zip(cols, row)) for row in res.fetchall()]

engine = mysql_engine()
print("Đã tạo engine MySQL. Sẵn sàng truy vấn.")


Đã tạo engine MySQL. Sẵn sàng truy vấn.


## 1) Nạp danh mục & mapping (authors, categories, publishers, books)

In [6]:

def load_catalog(engine):
    # Lấy vocab
    authors    = q_all(engine, f"SELECT id, name FROM {DB_BOOK}.authors")
    categories = q_all(engine, f"SELECT id, name FROM {DB_BOOK}.categories")
    publishers = q_all(engine, f"SELECT id, name FROM {DB_BOOK}.publishers")

    author2idx = {r["id"]: i for i, r in enumerate(authors)}
    cate2idx   = {r["id"]: i for i, r in enumerate(categories)}
    pub2idx    = {r["id"]: i for i, r in enumerate(publishers)}

    idx2author = {i: r["name"] for i, r in enumerate(authors)}
    idx2cate   = {i: r["name"] for i, r in enumerate(categories)}
    idx2pub    = {i: r["name"] for i, r in enumerate(publishers)}

    # Sách + publisher
    books = q_all(engine, f"SELECT id, title, publisher_id FROM {DB_BOOK}.books")
    book_ids = [r["id"] for r in books]
    book_meta = {r["id"]: {"title": r["title"], "publisher_id": r["publisher_id"]} for r in books}

    # Mapping N-N
    ab = q_all(engine, f"SELECT book_id, author_id FROM {DB_BOOK}.author_book")
    bc = q_all(engine, f"SELECT book_id, category_id FROM {DB_BOOK}.book_category")

    from collections import defaultdict
    book_authors = defaultdict(list)
    for r in ab:
        if r["author_id"] in author2idx:
            book_authors[r["book_id"]].append(author2idx[r["author_id"]])

    book_cates = defaultdict(list)
    for r in bc:
        if r["category_id"] in cate2idx:
            book_cates[r["book_id"]].append(cate2idx[r["category_id"]])

    book_pubs = {}
    for r in books:
        pid = r["publisher_id"]
        if pid in pub2idx:
            book_pubs[r["id"]] = pub2idx[pid]

    return {
        "authors": authors, "categories": categories, "publishers": publishers,
        "author2idx": author2idx, "cate2idx": cate2idx, "pub2idx": pub2idx,
        "idx2author": idx2author, "idx2cate": idx2cate, "idx2pub": idx2pub,
        "books": books, "book_ids": book_ids, "book_meta": book_meta,
        "book_authors": book_authors, "book_cates": book_cates, "book_pubs": book_pubs
    }

cat = load_catalog(engine)
print("Sách:", len(cat["book_ids"]), "| Tác giả:", len(cat["authors"]), "| Thể loại:", len(cat["categories"]), "| NXB:", len(cat["publishers"]))

Sách: 30 | Tác giả: 18 | Thể loại: 15 | NXB: 10


## 2) Xây ma trận **X** (Item–Feature) và **L2‑normalize** từng sách

In [7]:

# def build_X(catalog):
#     author2idx = catalog["author2idx"]
#     cate2idx   = catalog["cate2idx"]
#     pub2idx    = catalog["pub2idx"]
#     book_ids   = catalog["book_ids"]
#     book_authors = catalog["book_authors"]
#     book_cates   = catalog["book_cates"]
#     book_pubs    = catalog["book_pubs"]

#     # Số chiều từng nhóm & scale sqrt(lambda)
#     na, nc, npub = len(author2idx), len(cate2idx), len(pub2idx)
#     sa, sc, sp = math.sqrt(LAMBDA_AUTHOR), math.sqrt(LAMBDA_CATEGORY), math.sqrt(LAMBDA_PUBLISHER)
#     d = na + nc + npub

#     def item_vec(bid):
#         # Multi-hot cho tác giả/thể loại (chia đều 1/k), one-hot cho NXB
#         a_idx = book_authors.get(bid, [])
#         c_idx = book_cates.get(bid, [])
#         p_idx = book_pubs.get(bid, None)

#         va = np.zeros(na); vc = np.zeros(nc); vp = np.zeros(npub)
#         if a_idx:
#             fill = 1.0/len(a_idx)
#             for i in a_idx: va[i] = fill
#         if c_idx:
#             fill = 1.0/len(c_idx)
#             for i in c_idx: vc[i] = fill
#         if p_idx is not None:
#             vp[p_idx] = 1.0

#         # Nối 3 khối sau khi scale theo sqrt(lambda)
#         v = np.concatenate([sa*va, sc*vc, sp*vp])
#         # L2-normalize để tương đồng bằng cosine ổn định
#         n = np.linalg.norm(v)
#         return v/n if n>0 else v

#     X = np.zeros((len(book_ids), na+nc+npub))
#     for row_idx, bid in enumerate(book_ids):
#         X[row_idx, :] = item_vec(bid)
#     return X, na, nc, npub

# X, na, nc, npub = build_X(cat)
# book_ids = cat["book_ids"]
# book_meta = cat["book_meta"]
# print("X shape:", X.shape, "| na, nc, npub =", na, nc, npub)

# # Xem nhanh 10 sách đầu tiên (20 cột đầu)
# pd.DataFrame(
#     X[:PRINT_MAX_ITEMS, :min(na+nc+npub, 20)],
#     index=[f"{bid}:{book_meta[bid]['title']}" for bid in book_ids[:PRINT_MAX_ITEMS]]
# )



def build_X(catalog):
    author2idx = catalog["author2idx"]
    cate2idx   = catalog["cate2idx"]
    pub2idx    = catalog["pub2idx"]
    book_ids   = catalog["book_ids"]
    book_authors = catalog["book_authors"]
    book_cates   = catalog["book_cates"]
    book_pubs    = catalog["book_pubs"]

    # Số chiều từng nhóm & scale sqrt(lambda)
    na, nc, npub = len(author2idx), len(cate2idx), len(pub2idx)
    sa, sc, sp = math.sqrt(LAMBDA_AUTHOR), math.sqrt(LAMBDA_CATEGORY), math.sqrt(LAMBDA_PUBLISHER)
    d = na + nc + npub

    def item_vec(bid):
        # Multi-hot cho tác giả/thể loại (chia đều 1/k), one-hot cho NXB
        a_idx = book_authors.get(bid, [])
        c_idx = book_cates.get(bid, [])
        p_idx = book_pubs.get(bid, None)

        va = np.zeros(na); vc = np.zeros(nc); vp = np.zeros(npub)
        if a_idx:
            fill = 1.0/len(a_idx)
            for i in a_idx: va[i] = fill
        if c_idx:
            fill = 1.0/len(c_idx)
            for i in c_idx: vc[i] = fill
        if p_idx is not None:
            vp[p_idx] = 1.0

        v = np.concatenate([sa*va, sc*vc, sp*vp])
        n = np.linalg.norm(v)
        return v/n if n>0 else v

    # Ma trận X
    X = np.zeros((len(book_ids), d))
    for row_idx, bid in enumerate(book_ids):
        X[row_idx, :] = item_vec(bid)

    # === TÊN CỘT ĐẸP (chữ) ===
    feature_names = (
        [f"A:{catalog['idx2author'][i]}"   for i in range(na)] +
        [f"C:{catalog['idx2cate'][i]}"     for i in range(nc)] +
        [f"P:{catalog['idx2pub'][i]}"      for i in range(npub)]
    )

    return X, na, nc, npub, feature_names

X, na, nc, npub, FEATURE_NAMES = build_X(cat)
book_ids  = cat["book_ids"]
book_meta = cat["book_meta"]
print("X shape:", X.shape, "| na, nc, npub =", na, nc, npub)

# Xem nhanh 10 sách đầu tiên (20 cột đầu)
pd.DataFrame(
    X[:PRINT_MAX_ITEMS, :min(na+nc+npub, 20)],
    index=[f"{bid}:{book_meta[bid]['title']}" for bid in book_ids[:PRINT_MAX_ITEMS]],
    columns=FEATURE_NAMES[:min(na+nc+npub, 20)]
)

X shape: (30, 43) | na, nc, npub = 18 15 10


,A:Tô Hoài,A:Paulo CoeHo,A:Dale Carnegie,A:Rosie Nguyễn,A:Patrick Modiano,A:Hyun-wook park,A:Xuân Quỳnh,A:J. R. R. Tolkien,A:Diana Wynne Jones,A:Laura Cowan,A:Stephen Hawking,A: Sumino Yoru,A: Phương Hoài Nga,A: Alexandre Dumas,A:Chris Cooper,A:Trần Lỗi,A:Benjamin Graham,A:Raymond Chandler,C:Tiểu thuyết,C:Hài hước
1:Dế Mèn phưu lưu kí,0.707107,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.000000
2:Hoàng Tử Bé,0.000000,0.707107,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.353553
3:Đắc nhân tâm,0.000000,0.000000,0.632456,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
10:Công chúa ngủ trong rừng,0.632456,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
14:Tuổi trẻ đáng giá bao nhiêu?,0.000000,0.000000,0.000000,0.632456,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
15:Tuần trăng mật,0.000000,0.000000,0.000000,0.000000,0.707107,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.353553
16:Giã từ thơ ngây,0.000000,0.000000,0.000000,0.000000,0.000000,0.707107,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.000000
17:Hồi ức là cuộn băng tua ngược,0.000000,0.000000,0.000000,0.000000,0.000000,0.738549,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.246183,0.246183
18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.632456,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
19:KHÔNG BAO GIỜ LÀ CUỐI,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.632456,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000



## 3) Xây ma trận U 
- Nếu user có **nhiều review** một sách → **lấy review mới nhất** (`created_at DESC, id DESC`).  
- Nếu **vừa review vừa favorite** → dùng **điểm lớn hơn** (tức 5).


In [8]:

def build_W(engine, book_ids):
    book_set = set(book_ids)

    # Lấy review mới nhất cho mỗi (buyer, book)
    reviews_latest = q_all(engine, f'''
        SELECT buyer_id AS uid, book_id, stars AS rating
        FROM (
            SELECT buyer_id, book_id, stars,
                   ROW_NUMBER() OVER (PARTITION BY buyer_id, book_id ORDER BY created_at DESC, id DESC) AS rn
            FROM {DB_ORDER}.reviews
        ) t
        WHERE rn = 1
    ''')

    # Lấy favorites
    favs = q_all(engine, f"SELECT buyer_id AS uid, book_id FROM {DB_ORDER}.favorites")

    # Gom về uid -> {bid: score}
    from collections import defaultdict
    pairs_by_user = defaultdict(dict)

    # Nạp reviews
    for r in reviews_latest:
        if r["book_id"] in book_set:
            pairs_by_user[r["uid"]][r["book_id"]] = float(r["rating"])  # đúng thang 1..5

    # Nạp favorites: nếu chưa có review => 5; nếu có => lấy max(review,5)
    for f in favs:
        if f["book_id"] in book_set:
            cur = pairs_by_user[f["uid"]].get(f["book_id"], 0.0)
            pairs_by_user[f["uid"]][f["book_id"]] = max(cur, 5.0)

    # Xây W
    user_ids = sorted(pairs_by_user.keys())
    uid2row = {u:i for i, u in enumerate(user_ids)}
    m, n = len(user_ids), len(book_ids)
    W = np.zeros((m, n), dtype=float)

    for uid in user_ids:
        i = uid2row[uid]
        for bid, score in pairs_by_user[uid].items():
            j = book_ids.index(bid)
            W[i, j] = score  # giữ nguyên điểm 1..5 (favorite=5)

    return W, user_ids

W, user_ids = build_W(engine, book_ids)
print("W shape:", W.shape, "| users:", len(user_ids), "| items:", len(book_ids))

pd.DataFrame(
    W[:PRINT_MAX_USERS, :PRINT_MAX_ITEMS],
    index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
    columns=[f"b{j}:{book_meta[book_ids[j]]['title']}" for j in range(min(PRINT_MAX_ITEMS, len(book_ids)))]
)

W shape: (10, 30) | users: 10 | items: 30


,b0:Dế Mèn phưu lưu kí,b1:Hoàng Tử Bé,b2:Đắc nhân tâm,b3:Công chúa ngủ trong rừng,b4:Tuổi trẻ đáng giá bao nhiêu?,b5:Tuần trăng mật,b6:Giã từ thơ ngây,b7:Hồi ức là cuộn băng tua ngược,b8:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,b9:KHÔNG BAO GIỜ LÀ CUỐI
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u5:88875e62-c856-46fa-b705-fb8ee2fbf888,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0
u6:99975e62-c856-46fa-b705-fb8ee2fbf999,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0
u7:c8875e62-c856-46fa-b705-fb8ee2fbf44a,0.0,0.0,5.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0
u8:e32bf629-5b5e-4c92-9b1d-6f6d69a4909f,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,4.0,0.0
u9:fa3420ff-5fc4-4bcd-ac74-63d216d6247f,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



## 4) Tính **P = W · X** và **S = P̂ · Xᵀ** (cosine)  
- `P_raw = W·X` là hồ sơ thô.  
- Chuẩn hoá L2 từng dòng `P[u]` để có `P̂`.  
- `S = P̂ · Xᵀ` là điểm gợi ý (cosine với vector sách).


In [9]:

# def profiles_and_scores(W, X):
#     # P_raw[u] = sum_i W[u,i] * v_i (tổng có trọng số các vector sách mà user đã tương tác)
#     P_raw = W @ X

#     # Chuẩn hoá L2 từng user: p̂_u = p_u / ||p_u||_2
#     P = np.zeros_like(P_raw)
#     for i in range(P_raw.shape[0]):
#         nrm = np.linalg.norm(P_raw[i, :])
#         P[i, :] = P_raw[i, :] / nrm if nrm > 0 else P_raw[i, :]

#     # Điểm gợi ý: S[u,i] = p̂_u · v_i (vì v_i đã L2 -> cosine)
#     S = P @ X.T
#     return P, S

# P, S = profiles_and_scores(W, X)
# print("P shape:", P.shape, "| S shape:", S.shape)

# # Xem nhanh vài dòng đầu của P
# pd.DataFrame(P[:PRINT_MAX_USERS, :min(P.shape[1], 20)],
#              index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))])


def profiles_and_scores(W, X):
    """
    W: (m_users x n_items) điểm 1..5 (favorite=5).
    X: (n_items x d) vector sách đã L2.

    P_raw[u] = Σ_i W[u,i] * X[i]        (hồ sơ thô – tổng có trọng số)
    P[u]     = P_raw[u] / ||P_raw[u]||2 (chuẩn hoá L2 → vector đơn vị)
    S        = P · X^T                   (cosine vì X đã L2, P đã L2)
    """
    # 1) Hồ sơ thô (m x d)
    P_raw = W @ X

    # 2) Chuẩn hoá L2 từng user
    P = np.zeros_like(P_raw)
    for u in range(P_raw.shape[0]):
        nrm = np.linalg.norm(P_raw[u, :])
        P[u, :] = P_raw[u, :] / nrm if nrm > 0 else P_raw[u, :]

    # 3) Điểm gợi ý (m x n)
    S = P @ X.T
    return P, S, P_raw

P, S, P_raw = profiles_and_scores(W, X)
print("P shape:", P.shape, "| S shape:", S.shape)

# === In DataFrame với TÊN ĐẶC TRƯNG ===
n_users_show = min(PRINT_MAX_USERS, len(user_ids))
dfP = pd.DataFrame(
    P[:n_users_show, :],
    index=[f"u{i}:{user_ids[i]}" for i in range(n_users_show)],
    columns=FEATURE_NAMES
)
dfP_compact = dfP.loc[:, (dfP.abs() > 0).any(axis=0)]  # ẩn cột toàn 0 cho gọn
dfP_compact


P shape: (10, 43) | S shape: (10, 30)


,A:Tô Hoài,A:Paulo CoeHo,A:Dale Carnegie,A:Rosie Nguyễn,A:Patrick Modiano,A:Hyun-wook park,A:Xuân Quỳnh,A:J. R. R. Tolkien,A:Diana Wynne Jones,A:Laura Cowan,...,C:Trinh thám,P:Kim Đồng,P:Nhã Nam,P:NXB Trẻ,P:Hội nhà văn,P:Lao Động,P:Văn học,P:Dân Trí,P:Thế Giới,P:Phụ Nữ
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.522233,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
u1:44475e62-c856-46fa-b705-fb8ee2fbf444,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.422745,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.239141,0.000000,0.298926,0.000000,0.298926
u2:55575e62-c856-46fa-b705-fb8ee2fbf555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.430626,0.000000,0.000000,...,0.149925,0.000000,0.000000,0.000000,0.000000,0.000000,0.622537,0.000000,0.000000,0.000000
u3:66675e62-c856-46fa-b705-fb8ee2fbf666,0.441726,0.000000,0.000000,0.296319,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.312348,0.000000,0.209529,0.326236,0.000000,0.000000,0.000000,0.000000,0.000000
u4:77775e62-c856-46fa-b705-fb8ee2fbf777,0.000000,0.000000,0.411693,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.291111,0.000000,0.174667,0.000000,0.000000,0.325472,0.000000
u5:88875e62-c856-46fa-b705-fb8ee2fbf888,0.389249,0.000000,0.000000,0.000000,0.000000,0.454545,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.275241,0.000000,0.321412,0.000000,0.000000,0.000000,0.000000,0.000000,0.257130
u6:99975e62-c856-46fa-b705-fb8ee2fbf999,0.000000,0.000000,0.000000,0.000000,0.335847,0.000000,0.000000,0.000000,0.438476,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.547529,0.000000,0.000000,0.000000,0.000000,0.248040
u7:c8875e62-c856-46fa-b705-fb8ee2fbf44a,0.000000,0.000000,0.386955,0.000000,0.000000,0.259577,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.183549,0.273619,0.000000,0.319518,0.000000,0.000000,0.000000,0.000000
u8:e32bf629-5b5e-4c92-9b1d-6f6d69a4909f,0.000000,0.000000,0.000000,0.493865,0.000000,0.000000,0.395092,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.349215,0.279372,0.000000,0.000000,0.000000,0.000000,0.000000
u9:fa3420ff-5fc4-4bcd-ac74-63d216d6247f,0.000000,0.379629,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.396510,0.000000,...,0.000000,0.268439,0.000000,0.280375,0.280375,0.000000,0.000000,0.000000,0.000000,0.000000



## 5) Xuất  gợi ý
- Tập đã tương tác của user u chính là các cột có `W[u,i] > 0`.  
- Chỉ xếp hạng các item **chưa tương tác**.


In [10]:

def topk_recommend_for_user(u_idx, S, W, book_ids, book_meta, k=TOP_K):
    # Tập item user đã tương tác (đã review hoặc favorite)
    interacted = set(np.where(W[u_idx, :] > 0)[0])

    # Tính danh sách (j, score) cho item chưa tương tác
    cands = [(j, float(S[u_idx, j])) for j in range(len(book_ids)) if j not in interacted]
    cands.sort(key=lambda x: x[1], reverse=True)
    top = cands[:k]

    rows = []
    for j, sc in top:
        bid = book_ids[j]
        rows.append({"book_id": bid, "title": book_meta[bid]["title"], "score": sc})
    return pd.DataFrame(rows)

# In Top‑K cho vài user đầu tiên
for u in range(min(PRINT_MAX_USERS, len(user_ids))):
    print(f"\n=== User {u} ({user_ids[u]}) — Top {TOP_K} ===")
    display(topk_recommend_for_user(u, S, W, book_ids, book_meta, k=TOP_K))



=== User 0 (11175e62-c856-46fa-b705-fb8ee2fbf441) — Top 10 ===


,book_id,title,score
0,34,TÂY DU LẦY LỘI KÝ,1.000000
1,17,Hồi ức là cuộn băng tua ngược,0.333333
2,3,Đắc nhân tâm,0.233550
3,14,Tuổi trẻ đáng giá bao nhiêu?,0.233550
4,31,TRUYỆN TRANH BA CHÀNG LÍNH NGỰ LÂM,0.181818
5,2,Hoàng Tử Bé,0.087039
6,15,Tuần trăng mật,0.087039
7,23,LÂU ĐÀI BAY CỦA PHÁP SƯ HOWL,0.060606
8,24,LÂU ĐÀI TRÊN MÂY,0.060606
9,37,NGỦ GIẤC NGÀN THU,0.060606



=== User 1 (44475e62-c856-46fa-b705-fb8ee2fbf444) — Top 10 ===


,book_id,title,score
0,35,NUÔI CON KHÉO CHĂM CON NHÀN,0.662596
1,25,BẬT MÍ BÍ MẬT VỀ… VŨ TRỤ,0.634843
2,36,Nhà đầu tư thông minh,0.392139
3,27,MỞ KHÓA VŨ TRỤ,0.189320
4,32,MỌI ĐIỀU BẠN CẦN BIẾT VỀ VŨ TRỤ,0.189320
5,3,Đắc nhân tâm,0.160420
6,14,Tuổi trẻ đáng giá bao nhiêu?,0.160420
7,10,Công chúa ngủ trong rừng,0.089122
8,1,Dế Mèn phưu lưu kí,0.049821
9,2,Hoàng Tử Bé,0.000000



=== User 2 (55575e62-c856-46fa-b705-fb8ee2fbf555) — Top 10 ===


...

,book_id,title,score
0,36,Nhà đầu tư thông minh,0.523940
1,14,Tuổi trẻ đáng giá bao nhiêu?,0.462360
2,17,Hồi ức là cuộn băng tua ngược,0.398507
3,29,LÀM CHA MẸ HOÀN HẢO,0.169423
4,35,NUÔI CON KHÉO CHĂM CON NHÀN,0.169423
5,33,TAM QUỐC LẦY LỘI DIỄN NGHĨA,0.142893
6,34,TÂY DU LẦY LỘI KÝ,0.142893
7,23,LÂU ĐÀI BAY CỦA PHÁP SƯ HOWL,0.063904
8,24,LÂU ĐÀI TRÊN MÂY,0.063904
9,28,TỚ MUỐN ĂN TỤY CỦA CẬU,0.063904



=== User 8 (e32bf629-5b5e-4c92-9b1d-6f6d69a4909f) — Top 10 ===


,book_id,title,score
0,19,KHÔNG BAO GIỜ LÀ CUỐI,0.624695
1,3,Đắc nhân tâm,0.468521
2,17,Hồi ức là cuộn băng tua ngược,0.182372
3,33,TAM QUỐC LẦY LỘI DIỄN NGHĨA,0.182372
4,34,TÂY DU LẦY LỘI KÝ,0.182372
5,23,LÂU ĐÀI BAY CỦA PHÁP SƯ HOWL,0.145897
6,24,LÂU ĐÀI TRÊN MÂY,0.145897
7,28,TỚ MUỐN ĂN TỤY CỦA CẬU,0.145897
8,30,DẠI KHỜ ĐAU ĐỚN MONG MANH,0.145897
9,31,TRUYỆN TRANH BA CHÀNG LÍNH NGỰ LÂM,0.145897



=== User 9 (fa3420ff-5fc4-4bcd-ac74-63d216d6247f) — Top 10 ===


,book_id,title,score
0,23,LÂU ĐÀI BAY CỦA PHÁP SƯ HOWL,0.616144
1,34,TÂY DU LẦY LỘI KÝ,0.616144
2,15,Tuần trăng mật,0.367865
3,17,Hồi ức là cuộn băng tua ngược,0.337493
4,31,TRUYỆN TRANH BA CHÀNG LÍNH NGỰ LÂM,0.323302
5,28,TỚ MUỐN ĂN TỤY CỦA CẬU,0.258226
6,30,DẠI KHỜ ĐAU ĐỚN MONG MANH,0.258226
7,1,Dế Mèn phưu lưu kí,0.248058
8,16,Giã từ thơ ngây,0.160568
9,37,NGỦ GIẤC NGÀN THU,0.144343


## 6) Tương tự **item–item**: `Sim = X · Xᵀ` (cosine giữa các sách)

In [11]:

Sim = X @ X.T
pd.DataFrame(
    Sim[:PRINT_MAX_ITEMS, :PRINT_MAX_ITEMS],
    index=[f"{bid}:{book_meta[bid]['title']}" for bid in book_ids[:PRINT_MAX_ITEMS]],
    columns=[f"{bid}:{book_meta[bid]['title']}" for bid in book_ids[:PRINT_MAX_ITEMS]]
)

,1:Dế Mèn phưu lưu kí,2:Hoàng Tử Bé,3:Đắc nhân tâm,10:Công chúa ngủ trong rừng,14:Tuổi trẻ đáng giá bao nhiêu?,15:Tuần trăng mật,16:Giã từ thơ ngây,17:Hồi ức là cuộn băng tua ngược,18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,19:KHÔNG BAO GIỜ LÀ CUỐI
1:Dế Mèn phưu lưu kí,1.000000,0.375000,0.00000,0.894427,0.00000,0.125000,0.125000,0.087039,0.000000,0.000000
2:Hoàng Tử Bé,0.375000,1.000000,0.00000,0.223607,0.00000,0.250000,0.125000,0.174078,0.000000,0.000000
3:Đắc nhân tâm,0.000000,0.000000,1.00000,0.000000,0.60000,0.000000,0.000000,0.233550,0.000000,0.000000
10:Công chúa ngủ trong rừng,0.894427,0.223607,0.00000,1.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
14:Tuổi trẻ đáng giá bao nhiêu?,0.000000,0.000000,0.60000,0.000000,1.00000,0.000000,0.000000,0.233550,0.000000,0.000000
15:Tuần trăng mật,0.125000,0.250000,0.00000,0.000000,0.00000,1.000000,0.125000,0.174078,0.223607,0.223607
16:Giã từ thơ ngây,0.125000,0.125000,0.00000,0.000000,0.00000,0.125000,1.000000,0.696311,0.000000,0.000000
17:Hồi ức là cuộn băng tua ngược,0.087039,0.174078,0.23355,0.000000,0.23355,0.174078,0.696311,1.000000,0.000000,0.000000
18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,0.000000,0.000000,0.00000,0.000000,0.00000,0.223607,0.000000,0.000000,1.000000,1.000000
19:KHÔNG BAO GIỜ LÀ CUỐI,0.000000,0.000000,0.00000,0.000000,0.00000,0.223607,0.000000,0.000000,1.000000,1.000000


Lưu ma trận ra CSV để kiểm tra

In [12]:

# out_dir = "./cbf_outputs_simple_notebook"
# import os
# os.makedirs(out_dir, exist_ok=True)

# pd.DataFrame([{"book_id": bid, "title": book_meta[bid]["title"]} for bid in book_ids]).to_csv(f"{out_dir}/books.csv", index=False)
# pd.DataFrame({"user_id": user_ids}).to_csv(f"{out_dir}/users.csv", index=False)
# pd.DataFrame(W).to_csv(f"{out_dir}/W_user_item.csv", index=False)
# pd.DataFrame(X).to_csv(f"{out_dir}/X_item_feature.csv", index=False)
# pd.DataFrame(P).to_csv(f"{out_dir}/P_user_feature.csv", index=False)
# pd.DataFrame(S).to_csv(f"{out_dir}/S_scores.csv", index=False)

# print("Đã lưu vào:", os.path.abspath(out_dir))